# Model Comparison and Analysis

This notebook helps you compare different trained models and their performance.

## Features

- 📊 View all trained models and their metrics
- 🔍 Compare models side-by-side
- 📈 Visualize performance across experiments
- 🎯 Find the best model for your use case

In [ ]:
# Imports
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime

sns.set_style('whitegrid')
print("✓ Imports successful")

## 1. Load All Model Metadata

In [ ]:
def load_all_models():
    """
    Load metadata for all trained models.
    
    Returns:
    --------
    list of dict
        List of model metadata dictionaries
    """
    models_dir = Path('models')
    metadata_files = sorted(models_dir.glob('*_metadata.json'))
    
    models = []
    for metadata_file in metadata_files:
        with open(metadata_file, 'r') as f:
            models.append(json.load(f))
    
    return models

models = load_all_models()
print(f"Found {len(models)} trained models")

## 2. Create Comparison Table

In [ ]:
def create_comparison_table(models):
    """
    Create a pandas DataFrame comparing all models.
    
    Parameters:
    -----------
    models : list of dict
        List of model metadata
    
    Returns:
    --------
    pd.DataFrame
        Comparison table
    """
    rows = []
    for model in models:
        row = {
            'Model Name': model['model_name'],
            'Timestamp': model['timestamp'],
            'Window (s)': model['audio_params']['window_length'],
            'Accuracy': model['metrics']['accuracy'],
            'Precision': model['metrics']['precision'],
            'Recall': model['metrics']['recall'],
            'F1 Score': model['metrics']['f1_score'],
            'Best Epoch': model['metrics']['best_epoch'],
            'Val Loss': model['metrics']['best_val_loss'],
            'Epochs': model['training_params']['num_epochs'],
            'Batch Size': model['training_params']['batch_size'],
            'Learning Rate': model['training_params']['learning_rate']
        }
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df

comparison_df = create_comparison_table(models)

# Display with formatting
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print("\n" + "="*100)
print("MODEL COMPARISON TABLE")
print("="*100)
print(comparison_df.to_string(index=False))
print("="*100)

## 3. Find Best Models

In [ ]:
# Sort by different metrics
print("\n🏆 BEST MODELS BY METRIC")
print("="*80)

print("\n📊 Best Accuracy:")
best_acc = comparison_df.nlargest(3, 'Accuracy')[['Model Name', 'Window (s)', 'Accuracy', 'Recall']]
print(best_acc.to_string(index=False))

print("\n🎯 Best Recall (Most Important for Medical Use):")
best_recall = comparison_df.nlargest(3, 'Recall')[['Model Name', 'Window (s)', 'Recall', 'Accuracy']]
print(best_recall.to_string(index=False))

print("\n⚖️ Best F1 Score (Balanced):")
best_f1 = comparison_df.nlargest(3, 'F1 Score')[['Model Name', 'Window (s)', 'F1 Score', 'Accuracy', 'Recall']]
print(best_f1.to_string(index=False))

print("="*80)

## 4. Visualize Performance by Window Size

In [ ]:
# Plot metrics vs window size
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Accuracy vs Window Size
axes[0, 0].scatter(comparison_df['Window (s)'], comparison_df['Accuracy'], s=100, alpha=0.6)
axes[0, 0].set_xlabel('Window Length (seconds)')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].set_title('Accuracy vs Window Size')
axes[0, 0].grid(True, alpha=0.3)

# Recall vs Window Size
axes[0, 1].scatter(comparison_df['Window (s)'], comparison_df['Recall'], s=100, alpha=0.6, color='green')
axes[0, 1].set_xlabel('Window Length (seconds)')
axes[0, 1].set_ylabel('Recall')
axes[0, 1].set_title('Recall vs Window Size (Most Important!)')
axes[0, 1].grid(True, alpha=0.3)

# Precision vs Window Size
axes[1, 0].scatter(comparison_df['Window (s)'], comparison_df['Precision'], s=100, alpha=0.6, color='orange')
axes[1, 0].set_xlabel('Window Length (seconds)')
axes[1, 0].set_ylabel('Precision')
axes[1, 0].set_title('Precision vs Window Size')
axes[1, 0].grid(True, alpha=0.3)

# F1 Score vs Window Size
axes[1, 1].scatter(comparison_df['Window (s)'], comparison_df['F1 Score'], s=100, alpha=0.6, color='purple')
axes[1, 1].set_xlabel('Window Length (seconds)')
axes[1, 1].set_ylabel('F1 Score')
axes[1, 1].set_title('F1 Score vs Window Size')
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Compare Specific Models

In [ ]:
def compare_models(model_names):
    """
    Compare specific models in detail.
    
    Parameters:
    -----------
    model_names : list of str
        List of model names to compare
    """
    selected_models = [m for m in models if m['model_name'] in model_names]
    
    if not selected_models:
        print("No models found with those names")
        return
    
    print("\n" + "="*100)
    print("DETAILED MODEL COMPARISON")
    print("="*100)
    
    for model in selected_models:
        print(f"\n📊 {model['model_name']}")
        print("-" * 80)
        
        print("\nAudio Parameters:")
        for key, value in model['audio_params'].items():
            print(f"  {key:20s}: {value}")
        
        print("\nTraining Parameters:")
        for key, value in model['training_params'].items():
            print(f"  {key:20s}: {value}")
        
        print("\nPerformance Metrics:")
        for key, value in model['metrics'].items():
            if isinstance(value, float):
                print(f"  {key:20s}: {value:.4f}")
            else:
                print(f"  {key:20s}: {value}")
        
        print("\nConfusion Matrix:")
        cm = model['confusion_matrix']
        print(f"  True Negatives:  {cm['true_negatives']:5d}")
        print(f"  False Positives: {cm['false_positives']:5d}")
        print(f"  False Negatives: {cm['false_negatives']:5d}")
        print(f"  True Positives:  {cm['true_positives']:5d}")
        
        # Calculate additional metrics
        total = sum(cm.values())
        false_negative_rate = cm['false_negatives'] / (cm['false_negatives'] + cm['true_positives'])
        false_positive_rate = cm['false_positives'] / (cm['false_positives'] + cm['true_negatives'])
        
        print(f"\n  ⚠️ False Negative Rate: {false_negative_rate:.2%} (missed alarms)")
        print(f"  ⚠️ False Positive Rate: {false_positive_rate:.2%} (false alarms)")
    
    print("\n" + "="*100)

# Example: Compare first two models (if they exist)
if len(models) >= 2:
    compare_models([models[0]['model_name'], models[1]['model_name']])
elif len(models) == 1:
    compare_models([models[0]['model_name']])
else:
    print("No models to compare yet. Train some models first!")

## 6. View Training History

In [ ]:
def plot_training_history(model_name):
    """
    Plot training curves for a specific model.
    
    Parameters:
    -----------
    model_name : str
        Name of the model to plot
    """
    model = next((m for m in models if m['model_name'] == model_name), None)
    
    if not model:
        print(f"Model '{model_name}' not found")
        return
    
    history = model['training_history']
    epochs = range(1, len(history['train_losses']) + 1)
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    
    # Loss curves
    axes[0].plot(epochs, history['train_losses'], label='Train Loss', marker='o')
    axes[0].plot(epochs, history['val_losses'], label='Val Loss', marker='o')
    axes[0].axvline(x=model['metrics']['best_epoch'], color='r', linestyle='--', 
                    label=f"Best Epoch ({model['metrics']['best_epoch']})")
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Loss')
    axes[0].set_title(f'Training History - {model_name}')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    # Accuracy curve
    axes[1].plot(epochs, history['val_accuracies'], label='Val Accuracy', marker='o', color='green')
    axes[1].axvline(x=model['metrics']['best_epoch'], color='r', linestyle='--',
                    label=f"Best Epoch ({model['metrics']['best_epoch']})")
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Accuracy')
    axes[1].set_title('Validation Accuracy')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.show()

# Plot training history for the first model
if len(models) > 0:
    plot_training_history(models[0]['model_name'])
else:
    print("No models to plot yet. Train some models first!")

## 7. Export Comparison Report

In [ ]:
# Save comparison table to CSV
output_path = Path('models') / 'model_comparison.csv'
comparison_df.to_csv(output_path, index=False)
print(f"✓ Comparison table saved to: {output_path}")

# Create a summary report
report_path = Path('models') / 'model_comparison_report.txt'
with open(report_path, 'w') as f:
    f.write("="*100 + "\n")
    f.write("MODEL COMPARISON REPORT\n")
    f.write(f"Generated: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("="*100 + "\n\n")
    
    f.write(f"Total Models Trained: {len(models)}\n\n")
    
    f.write("BEST MODELS:\n")
    f.write("-" * 80 + "\n")
    
    best_acc_model = comparison_df.loc[comparison_df['Accuracy'].idxmax()]
    f.write(f"\nBest Accuracy: {best_acc_model['Accuracy']:.4f}\n")
    f.write(f"  Model: {best_acc_model['Model Name']}\n")
    f.write(f"  Window: {best_acc_model['Window (s)']}s\n")
    
    best_recall_model = comparison_df.loc[comparison_df['Recall'].idxmax()]
    f.write(f"\nBest Recall: {best_recall_model['Recall']:.4f}\n")
    f.write(f"  Model: {best_recall_model['Model Name']}\n")
    f.write(f"  Window: {best_recall_model['Window (s)']}s\n")
    
    best_f1_model = comparison_df.loc[comparison_df['F1 Score'].idxmax()]
    f.write(f"\nBest F1 Score: {best_f1_model['F1 Score']:.4f}\n")
    f.write(f"  Model: {best_f1_model['Model Name']}\n")
    f.write(f"  Window: {best_f1_model['Window (s)']}s\n")
    
    f.write("\n" + "="*100 + "\n")
    f.write("\nFULL COMPARISON TABLE:\n")
    f.write("-" * 80 + "\n")
    f.write(comparison_df.to_string(index=False))

print(f"✓ Summary report saved to: {report_path}")